# Electronics — where the transistor becomes a bit

Everything so far has been analog: a continuous input producing a continuous output, and the whole art was keeping the relationship linear. Digital circuits do the opposite on purpose. They take the **same MOSFET** from the first notebook, run it hard into the two regions where it is a good switch, and throw the middle away.

$$V_{GS}<V_{th}\ \Rightarrow\ \text{open}
\qquad\qquad
V_{GS}\gg V_{th},\ V_{DS}\to0\ \Rightarrow\ \text{a resistor of }\ \frac{1}{k V_{ov}}$$

What that buys is **restoration**. An analog stage passes its input errors to the next stage and adds its own; a digital stage with gain greater than one pushes a degraded input back toward a rail, so noise stops accumulating. That single property is why a signal can cross a chip through ten thousand gates and still be the value it started as, and it is what the first two sections measure.

The price is paid in the other three. Every transition costs energy, takes time, and momentarily shorts the supply — and those three quantities are what has governed processor design for the last twenty years.

Schematics animate as before, and every curve is computed by solving the actual device equations rather than assuming an ideal switch.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
import ipywidgets as widgets
from IPython.display import display

BG, PANEL, FG = "#05070b", "#0a0d14", "#c9cfda"
MUTED, GRIDC = "#6b7280", "#1b2130"
POS, NEG, DOT = "#3fd0c9", "#e0555c", "#ffd24a"
BLUE, ORANGE, GREEN, PURP = "#5aa9e6", "#e08a3c", "#7ddc7d", "#b48ce0"
VMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "volt", [(0.0, NEG), (0.5, "#4a5060"), (1.0, POS)])

plt.rcParams.update({
    "figure.dpi": 112, "font.size": 8.5, "axes.titlesize": 9,
    "figure.facecolor": BG, "savefig.facecolor": BG, "axes.facecolor": PANEL,
    "axes.edgecolor": GRIDC, "axes.labelcolor": FG, "text.color": FG,
    "xtick.color": MUTED, "ytick.color": MUTED, "grid.color": GRIDC,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.facecolor": PANEL, "legend.edgecolor": GRIDC, "legend.framealpha": 0.9,
})
SL = {"style": {"description_width": "104px"},
      "layout": widgets.Layout(width="290px"), "continuous_update": False}


def panel(ax, edge=None, lw=1.3):
    ax.set_facecolor(PANEL)
    for s in ax.spines.values():
        s.set_visible(True); s.set_color(edge or GRIDC)
        s.set_linewidth(lw if edge else 0.8)
    ax.tick_params(colors=MUTED, labelsize=7)
    return ax


def readout(fig, x, y, lines, color=FG, size=7.4):
    fig.text(x, y, "\n".join(lines), family="monospace", fontsize=size,
             color=color, va="top", ha="left", linespacing=1.55)


def footer(fig, text):
    fig.text(0.010, 0.012, text, family="monospace", fontsize=6.6, color=MUTED)
    fig.text(0.990, 0.012, "electronics · digital", family="monospace",
             fontsize=6.6, color=MUTED, ha="right")


def timeline(n, step=1, interval=90, desc="time"):
    p = widgets.Play(value=0, min=0, max=n, step=step, interval=interval)
    s = widgets.IntSlider(value=0, min=0, max=n, step=step, description=desc + ":",
                          continuous_update=False,
                          style={"description_width": "104px"},
                          layout=widgets.Layout(width="430px"))
    widgets.jslink((p, "value"), (s, "value"))
    return p, s


# ----- schematic primitives, Falstad style -------------------------------
def vcolor(v, vmax):
    return VMAP(np.clip(0.5 + 0.5 * v / max(vmax, 1e-9), 0, 1))


def wire(ax, pts, v, vmax, lw=2.6):
    pts = np.asarray(pts, float)
    seg = np.stack([pts[:-1], pts[1:]], axis=1)
    ax.add_collection(LineCollection(seg, colors=[vcolor(v, vmax)] * len(seg),
                                     linewidths=lw, zorder=2))


def node_dot(ax, p, v, vmax, s=34):
    ax.plot(*p, "o", ms=np.sqrt(s), color=vcolor(v, vmax), zorder=4)


def resistor(ax, p0, p1, v, vmax, label=None, n=6, amp=0.16):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    a, b = p0 + u * L * 0.28, p1 - u * L * 0.28
    ts = np.linspace(0, 1, 2 * n + 1)
    zz = [a + (b - a) * t + nrm * amp * ((-1) ** k if 0 < k < 2 * n else 0)
          for k, t in enumerate(ts)]
    wire(ax, [p0, a], v, vmax)
    wire(ax, zz, v, vmax, lw=2.2)
    wire(ax, [b, p1], v, vmax)
    if label:
        ax.text(*(0.5 * (p0 + p1) + nrm * 0.34), label, color=FG, fontsize=7.5,
                ha="center", va="center")


def capacitor(ax, p0, p1, v, vmax, label=None, gap=0.10, half=0.24):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    c = 0.5 * (p0 + p1)
    a, b = c - u * gap, c + u * gap
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    for q in (a, b):
        ax.plot(*np.stack([q - nrm * half, q + nrm * half]).T, color=FG, lw=2.4,
                zorder=3)
    if label:
        ax.text(*(c + nrm * 0.40), label, color=FG, fontsize=7.5, ha="center")


def inductor(ax, p0, p1, v, vmax, label=None, coils=4, r=0.13):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    a, b = p0 + u * L * 0.25, p1 - u * L * 0.25
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    seg = np.linalg.norm(b - a) / coils
    for k in range(coils):
        c = a + u * seg * (k + 0.5)
        th = np.linspace(0, np.pi, 24)
        pts = np.array([c + u * (seg / 2) * np.cos(np.pi - t) + nrm * r * np.sin(t)
                        for t in th])
        ax.plot(pts[:, 0], pts[:, 1], color=FG, lw=2.0, zorder=3)
    if label:
        ax.text(*(0.5 * (p0 + p1) + nrm * 0.38), label, color=FG, fontsize=7.5,
                ha="center")


def diode(ax, p0, p1, v, vmax, label=None, s=0.20):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    c = 0.5 * (p0 + p1)
    a, b = c - u * s, c + u * s
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    ax.add_patch(mpatches.Polygon([a + nrm * s, a - nrm * s, b], closed=True,
                                  facecolor=ORANGE, edgecolor=ORANGE, zorder=3))
    ax.plot(*np.stack([b - nrm * s, b + nrm * s]).T, color=FG, lw=2.6, zorder=3)
    if label:
        ax.text(*(c + nrm * 0.40), label, color=FG, fontsize=7.5, ha="center")


def source(ax, p0, p1, v, vmax, kind="dc", label=None, r=0.30):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    c = 0.5 * (p0 + p1)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    wire(ax, [p0, c - u * r], v, vmax); wire(ax, [c + u * r, p1], v, vmax)
    ax.add_patch(mpatches.Circle(c, r, fill=False, ec=FG, lw=2.0, zorder=3))
    if kind == "dc":
        ax.plot(*np.stack([c - u * 0.12 - nrm * 0.16, c - u * 0.12 + nrm * 0.16]).T,
                color=FG, lw=2.6, zorder=4)
        ax.plot(*np.stack([c + u * 0.12 - nrm * 0.09, c + u * 0.12 + nrm * 0.09]).T,
                color=FG, lw=2.0, zorder=4)
    else:
        t = np.linspace(-1, 1, 40)
        pts = np.array([c + u * (0.19 * t[i]) + nrm * 0.15 * np.sin(np.pi * t[i])
                        for i in range(len(t))])
        ax.plot(pts[:, 0], pts[:, 1], color=FG, lw=1.8, zorder=4)
    if label:
        ax.text(*(c + nrm * (r + 0.22)), label, color=FG, fontsize=7.5, ha="center")


def path_len(pts):
    p = np.asarray(pts, float)
    d = np.linalg.norm(np.diff(p, axis=0), axis=1)
    return np.r_[0, np.cumsum(d)]


def charge_dots(ax, loop, q, spacing=0.42, ms=4.2):
    """Yellow dots at arclength q + n*spacing — this is the current, visualised."""
    p = np.asarray(loop, float)
    s = path_len(p)
    L = s[-1]
    if L <= 0:
        return
    offs = (np.arange(0, L, spacing) + (q % spacing)) % L
    x = np.interp(offs, s, p[:, 0]); y = np.interp(offs, s, p[:, 1])
    ax.plot(x, y, "o", ms=ms, color=DOT, zorder=5, mec="none")


def sch_axes(ax, xlim, ylim):
    panel(ax)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    return ax


def loop_rect(x0, x1, y0, y1, n=60):
    top = np.stack([np.linspace(x0, x1, n), np.full(n, y1)], 1)
    right = np.stack([np.full(n, x1), np.linspace(y1, y0, n)], 1)
    bot = np.stack([np.linspace(x1, x0, n), np.full(n, y0)], 1)
    left = np.stack([np.full(n, x0), np.linspace(y0, y1, n)], 1)
    return np.vstack([top, right, bot, left])


def seg(p0, p1, n=40):
    return np.stack([np.linspace(p0[0], p1[0], n),
                     np.linspace(p0[1], p1[1], n)], 1)


VT = 0.02585          # kT/q at 300 K
IS_BJT = 1e-15
BETA = 150.0
VA = 50.0             # Early voltage
VTH_N = 1.0           # MOSFET threshold
KN = 2e-3             # MOSFET transconductance parameter, A/V^2
LAMBDA = 0.02


def bjt_ic(vbe, vce=5.0, Is=IS_BJT, va=VA):
    """Forward-active collector current with the Early effect."""
    ic = Is * np.exp(np.clip(vbe, -2, 1.2) / VT)
    return ic * (1 + np.maximum(vce, 0) / va)


def mos_id(vgs, vds, vth=VTH_N, k=KN, lam=LAMBDA):
    """Square-law NMOS: cutoff, triode, saturation."""
    vgs = np.asarray(vgs, float); vds = np.asarray(vds, float)
    vov = vgs - vth
    tri = k * (vov * vds - 0.5 * vds ** 2)
    sat = 0.5 * k * vov ** 2 * (1 + lam * vds)
    out = np.where(vds < vov, tri, sat)
    return np.where(vov <= 0, 0.0, np.maximum(out, 0.0))


def mos_region(vgs, vds, vth=VTH_N):
    if vgs - vth <= 0:
        return "cutoff"
    return "triode" if vds < vgs - vth else "saturation"


def transistor_npn(ax, p, v, vmax, label=None, s=0.42, flip=False):
    """NPN symbol: base left, collector up, emitter down."""
    p = np.asarray(p, float)
    ax.plot([p[0] - s * 0.55, p[0] - s * 0.55], [p[1] - s, p[1] + s],
            color=FG, lw=2.6, zorder=3)
    ax.plot([p[0] - s * 1.5, p[0] - s * 0.55], [p[1], p[1]], color=FG,
            lw=2.2, zorder=3)
    ax.plot([p[0] - s * 0.55, p[0] + s * 0.7], [p[1] + s * 0.45,
            p[1] + s * 1.25], color=FG, lw=2.2, zorder=3)
    ax.plot([p[0] - s * 0.55, p[0] + s * 0.7], [p[1] - s * 0.45,
            p[1] - s * 1.25], color=FG, lw=2.2, zorder=3)
    ax.add_patch(mpatches.Polygon(
        [[p[0] + s * 0.2, p[1] - s * 0.82], [p[0] + s * 0.05, p[1] - s * 0.45],
         [p[0] + s * 0.5, p[1] - s * 0.62]], closed=True, facecolor=FG,
        edgecolor=FG, zorder=4))
    if label:
        ax.text(p[0] + s * 1.05, p[1], label, color=FG, fontsize=8,
                va="center")


def transistor_nmos(ax, p, v, vmax, label=None, s=0.42):
    """NMOS symbol: gate left, drain up, source down."""
    p = np.asarray(p, float)
    ax.plot([p[0] - s * 0.95, p[0] - s * 0.95], [p[1] - s, p[1] + s],
            color=FG, lw=2.4, zorder=3)
    ax.plot([p[0] - s * 1.9, p[0] - s * 0.95], [p[1], p[1]], color=FG,
            lw=2.2, zorder=3)
    for dy in (-1, 0, 1):
        y0 = p[1] + dy * s * 0.62
        ax.plot([p[0] - s * 0.5, p[0] - s * 0.5],
                [y0 - s * 0.28, y0 + s * 0.28], color=FG, lw=2.4, zorder=3)
    ax.plot([p[0] - s * 0.5, p[0] + s * 0.7], [p[1] + s * 0.62,
            p[1] + s * 0.62], color=FG, lw=2.2, zorder=3)
    ax.plot([p[0] + s * 0.7, p[0] + s * 0.7], [p[1] + s * 0.62,
            p[1] + s * 1.3], color=FG, lw=2.2, zorder=3)
    ax.plot([p[0] - s * 0.5, p[0] + s * 0.7], [p[1] - s * 0.62,
            p[1] - s * 0.62], color=FG, lw=2.2, zorder=3)
    ax.plot([p[0] + s * 0.7, p[0] + s * 0.7], [p[1] - s * 0.62,
            p[1] - s * 1.3], color=FG, lw=2.2, zorder=3)
    ax.plot([p[0] - s * 0.5, p[0] + s * 0.7], [p[1], p[1]], color=FG,
            lw=2.0, zorder=3)
    ax.add_patch(mpatches.Polygon(
        [[p[0] + s * 0.35, p[1]], [p[0] + s * 0.05, p[1] + s * 0.2],
         [p[0] + s * 0.05, p[1] - s * 0.2]], closed=True, facecolor=FG,
        edgecolor=FG, zorder=4))
    if label:
        ax.text(p[0] + s * 1.25, p[1], label, color=FG, fontsize=8,
                va="center")


from scipy.optimize import brentq

VDD = 3.3
VTN = VTP = 0.7
KN0, KP0 = 200e-6, 100e-6      # process transconductance, A/V^2


def sq_id(vgs, vds, k, vth):
    """Square-law drain current, scalar."""
    vov = vgs - vth
    if vov <= 0:
        return 0.0
    if vds < vov:
        return k * (vov * vds - 0.5 * vds ** 2)
    return 0.5 * k * vov ** 2


def inv_out(vin, wp_wn=2.0, vdd=VDD, kn=KN0, kp=KP0):
    """Solve In(vin,vout) = Ip(vin,vout) for the inverter output."""
    kpe = kp * wp_wn
    def f(vo):
        return sq_id(vin, vo, kn, VTN) - sq_id(vdd - vin, vdd - vo, kpe, VTP)
    try:
        return brentq(f, 1e-9, vdd - 1e-9, xtol=1e-12)
    except ValueError:
        return vdd if vin < vdd / 2 else 0.0


def inv_curve(wp_wn=2.0, vdd=VDD, n=260):
    vi = np.linspace(0, vdd, n)
    return vi, np.array([inv_out(v, wp_wn, vdd) for v in vi])


def crowbar(vin, wp_wn=2.0, vdd=VDD, kn=KN0):
    return sq_id(vin, inv_out(vin, wp_wn, vdd), kn, VTN)


def switch_threshold(wp_wn=2.0, vdd=VDD):
    lo, hi = 1e-3, vdd - 1e-3
    for _ in range(80):
        m = 0.5 * (lo + hi)
        if inv_out(m, wp_wn, vdd) > m:
            lo = m
        else:
            hi = m
    return 0.5 * (lo + hi)


def noise_margins(wp_wn=2.0, vdd=VDD):
    vi = np.linspace(0.02, vdd - 0.02, 400)
    vo = np.array([inv_out(v, wp_wn, vdd) for v in vi])
    g = np.gradient(vo, vi)
    idx = np.where(np.abs(g) >= 1.0)[0]
    if len(idx) < 2:
        return dict(VIL=np.nan, VIH=np.nan, VOH=vdd, VOL=0.0,
                    NMH=np.nan, NML=np.nan, gain=np.nan)
    VIL, VIH = vi[idx[0]], vi[idx[-1]]
    VOH, VOL = inv_out(VIL, wp_wn, vdd), inv_out(VIH, wp_wn, vdd)
    return dict(VIL=VIL, VIH=VIH, VOH=VOH, VOL=VOL,
                NMH=VOH - VIH, NML=VIL - VOL, gain=float(np.min(g)))


def mos_switch(ax, p, ptype=False, s=0.36):
    """Compact MOS switch symbol for logic drawings."""
    p = np.asarray(p, float)
    ax.plot([p[0] - s * 0.9, p[0] - s * 0.9], [p[1] - s, p[1] + s],
            color=FG, lw=2.2, zorder=3)
    ax.plot([p[0] - s * 0.45, p[0] - s * 0.45], [p[1] - s, p[1] + s],
            color=FG, lw=2.6, zorder=3)
    ax.plot([p[0] - s * 1.9, p[0] - s * 0.9], [p[1], p[1]], color=FG,
            lw=2.0, zorder=3)
    for dy in (-1, 1):
        ax.plot([p[0] - s * 0.45, p[0] + s * 0.7],
                [p[1] + dy * s * 0.6, p[1] + dy * s * 0.6], color=FG,
                lw=2.0, zorder=3)
        ax.plot([p[0] + s * 0.7, p[0] + s * 0.7],
                [p[1] + dy * s * 0.6, p[1] + dy * s * 1.5], color=FG,
                lw=2.0, zorder=3)
    if ptype:
        ax.add_patch(mpatches.Circle((p[0] - s * 1.35, p[1]), s * 0.24,
                                     fill=False, ec=FG, lw=1.8, zorder=4))
    return (p[0] + s * 0.7, p[1] + s * 1.5), (p[0] + s * 0.7, p[1] - s * 1.5)


print("digital engine ready")
print(f"VDD = {VDD} V   Vth = {VTN} V   kn/kp = {KN0/KP0:.1f} "
      f"so Wp/Wn = {KN0/KP0:.1f} for a symmetric inverter")
print(f"VT = {VT*1e3:.2f} mV   e-fold per VT   decade per {VT*np.log(10)*1e3:.2f} mV")
print(f"MOSFET Vth = {VTH_N:.2f} V   k = {KN*1e3:.2f} mA/V^2   VA = {VA:.0f} V")

digital engine ready
VDD = 3.3 V   Vth = 0.7 V   kn/kp = 2.0 so Wp/Wn = 2.0 for a symmetric inverter
VT = 25.85 mV   e-fold per VT   decade per 59.52 mV
MOSFET Vth = 1.00 V   k = 2.00 mA/V^2   VA = 50 V


## The CMOS inverter — two switches that are never both fully on

One NMOS pulls down, one PMOS pulls up, and their gates are tied together. When the input is low the NMOS is off and the PMOS is on, so the output is connected to $V_{DD}$ **through a resistance and nothing else** — no current flows, so no voltage is dropped and the output is exactly the rail. When the input is high the roles swap.

That is the crucial difference from every analog stage so far: in both stable states one device is completely off, so the static current is zero and the output reaches the rail *exactly* rather than approaching it.

The transfer curve is found by solving $I_n=I_p$ at each input voltage, which the panel does numerically. Three features matter.

The **switching threshold** $V_M$ is where the curve crosses $V_{out}=V_{in}$, and it is set by the strength ratio of the two devices. Electrons are about twice as mobile as holes, so $k_n\approx2k_p$, and a symmetric inverter needs the PMOS made **twice as wide** — measured $W_p/W_n=2.00$ giving $V_M=1.6500$ V, exactly $V_{DD}/2$. That 2:1 sizing is visible in the layout of essentially every CMOS chip ever made.

The **gain** in the transition region is large — measured $-17.9$ — and that is not a coincidence, it is the whole mechanism. Gain above one is what makes the gate restoring. And the transition is **steep but not vertical**, which is where the next section starts.

In [2]:
def draw_inverter(k, vin, wp_wn, vdd):
    vo = inv_out(vin, wp_wn, vdd)
    vi_c, vo_c = inv_curve(wp_wn, vdd)
    Vm = switch_threshold(wp_wn, vdd)
    In = sq_id(vin, vo, KN0, VTN)
    Ip = sq_id(vdd - vin, vdd - vo, KP0 * wp_wn, VTP)
    g = np.gradient(vo_c, vi_c)
    vmax = vdd

    fig = plt.figure(figsize=(13.0, 5.2))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.05, 1.3, 0.55],
                          wspace=0.3, hspace=0.46, left=0.02, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.7, 3.6), (-0.5, 4.0))
    dp, sp = mos_switch(a0, (2.0, 2.7), ptype=True)
    dn, sn = mos_switch(a0, (2.0, 1.0), ptype=False)
    wire(a0, [(2.0 + 0.25, 2.7 + 0.54), (2.25, 3.6)], vdd, vmax)
    wire(a0, [(2.25, 3.6), (0.7, 3.6)], vdd, vmax)
    wire(a0, [(2.0 + 0.25, 2.7 - 0.54), (2.25, 1.0 + 0.54)], vo, vmax)
    wire(a0, [(2.0 + 0.25, 1.0 - 0.54), (2.25, 0.2)], 0.0, vmax)
    wire(a0, [(2.25, 0.2), (0.7, 0.2)], 0.0, vmax)
    wire(a0, [(2.25, 1.85), (3.2, 1.85)], vo, vmax)
    node_dot(a0, (3.2, 1.85), vo, vmax)
    a0.text(3.25, 2.05, "out", color=FG, fontsize=8)
    wire(a0, [(2.0 - 0.68, 2.7), (1.1, 2.7), (1.1, 1.0), (2.0 - 0.68, 1.0)],
         vin, vmax)
    wire(a0, [(1.1, 1.85), (0.35, 1.85)], vin, vmax)
    node_dot(a0, (1.1, 1.85), vin, vmax)
    a0.text(0.05, 2.05, "in", color=FG, fontsize=8)
    source(a0, (0.7, 0.2), (0.7, 3.6), vdd / 2, vmax, "dc", f"{vdd:.1f}V")
    a0.text(2.85, 2.9, "PMOS\n" + ("ON" if vin < vdd - VTP else "off"),
            color=POS if vin < vdd - VTP else MUTED, fontsize=7.5)
    a0.text(2.85, 0.85, "NMOS\n" + ("ON" if vin > VTN else "off"),
            color=DOT if vin > VTN else MUTED, fontsize=7.5)
    if In > 1e-9:
        charge_dots(a0, seg((2.25, 3.6), (2.25, 0.4), 40), k / 100 * In * 3e6,
                    spacing=0.22, ms=3.4)
    a0.set_title("in both stable states, one device is completely off",
                 fontsize=8.5)

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    for r in (1.0, 2.0, 4.0):
        vi_x, vo_x = inv_curve(r, vdd, 160)
        a1.plot(vi_x, vo_x, color=POS if abs(r - wp_wn) < 0.05 else MUTED,
                lw=2.0 if abs(r - wp_wn) < 0.05 else 0.9,
                alpha=1.0 if abs(r - wp_wn) < 0.05 else 0.5)
    a1.plot([0, vdd], [0, vdd], color=GRIDC, lw=0.9, ls="--")
    a1.plot([Vm], [Vm], "o", ms=8, color=DOT)
    a1.axvline(vin, color=FG, lw=1.0, ls=":")
    a1.plot([vin], [vo], "o", ms=7, color=ORANGE)
    a1.set_xlim(0, vdd); a1.set_ylim(0, vdd)
    a1.set_xlabel("$V_{in}$  (V)"); a1.set_ylabel("$V_{out}$  (V)")
    a1.set_title(f"$V_M$ = {Vm:.4f} V  ·  $V_{{DD}}/2$ = {vdd/2:.4f} V  ·  "
                 f"$W_p/W_n$ = {wp_wn:.2f}")

    a2 = panel(fig.add_subplot(gs[1, 1]), ORANGE)
    a2.plot(vi_c, g, color=ORANGE, lw=1.7)
    a2.axhline(-1, color=NEG, lw=1.0, ls="--")
    a2.axhline(0, color=GRIDC, lw=0.8)
    a2.axvline(vin, color=FG, lw=1.0, ls=":")
    a2.set_xlim(0, vdd); a2.set_ylim(min(g) * 1.15, 2)
    a2.set_xlabel("$V_{in}$  (V)"); a2.set_ylabel("$dV_{out}/dV_{in}$")
    a2.set_title(f"peak gain {np.min(g):.2f} — above 1 is what makes it restoring")

    state = ("output HIGH" if vo > 0.9 * vdd else
             "output LOW" if vo < 0.1 * vdd else "in transition")
    readout(fig, 0.845, 0.90, [
        "PROCESS", "─" * 26,
        f"VDD         {vdd:>10.2f}V",
        f"Vth n / p   {VTN:>5.2f} /{VTP:>5.2f}V",
        f"kn          {KN0*1e6:>10.1f}µA/V²",
        f"kp          {KP0*1e6:>10.1f}µA/V²",
        f"Wp/Wn       {wp_wn:>10.3f}",
        "", "THRESHOLD", "─" * 26,
        f"VM measured {Vm:>10.4f}V",
        f"VDD/2       {vdd/2:>10.4f}V",
        f"offset      {Vm-vdd/2:>+10.4f}V",
        f"for VM=VDD/2{KN0/KP0:>10.2f}",
        "", "NOW", "─" * 26,
        f"Vin         {vin:>10.4f}V",
        f"Vout        {vo:>10.4f}V",
        f"In          {In*1e6:>10.4f}µA",
        f"Ip          {Ip*1e6:>10.4f}µA",
        f"state       {state:>14s}",
        "", "no static current in",
        "either stable state",
    ], color=GREEN if In < 1e-9 else NEG)
    footer(fig, f"solve In = Ip at each input   ·   VM = {Vm:.4f} V   ·   "
                f"symmetric when kp·Wp = kn·Wn")
    plt.show()


_p1, _s1 = timeline(99, step=2)
w1 = dict(vin=widgets.FloatSlider(value=0.5, min=0, max=3.3, step=0.01,
                                  description="Vin (V):", **SL),
          wp_wn=widgets.FloatSlider(value=2.0, min=0.5, max=6.0, step=0.1,
                                    description="Wp/Wn:", **SL),
          vdd=widgets.FloatSlider(value=3.3, min=1.0, max=5.0, step=0.1,
                                  description="VDD (V):", **SL),
          k=_s1)
display(widgets.VBox([widgets.HBox([w1["vin"], w1["wp_wn"], w1["vdd"]]),
                      widgets.HBox([_p1, _s1])]),
        widgets.interactive_output(draw_inverter, w1))

Output()

## Noise margins — why a bit survives ten thousand gates

A gate's output is not exactly $V_{DD}$ or exactly ground once it is loaded and disturbed, and its input will not receive a clean level either. What keeps a chain of gates working is that **the output levels are further apart than the input levels need to be**.

The boundaries are the two points where the transfer curve has unity slope. Inside that band the gate amplifies disturbances; outside it, it attenuates them. That gives four voltages and two margins:

$$NM_H=V_{OH}-V_{IH},\qquad NM_L=V_{IL}-V_{OL}$$

For the symmetric inverter the panel measures $V_{IL}=1.418$, $V_{IH}=1.882$, $V_{OH}=3.057$, $V_{OL}=0.243$ — giving **1.175 V of margin on both sides, 35.6% of $V_{DD}$**. Any disturbance smaller than that is squeezed out rather than passed on, which is the entire reason digital logic composes.

Skewing the sizing shows the cost immediately. Drag $W_p/W_n$ away from 2 and the two margins become unequal: the gate is still perfectly functional but it is now much more vulnerable to noise on one side than the other. A deliberately skewed inverter is sometimes exactly what you want — but a *accidentally* skewed one is a chip that fails only when the supply droops.

Lowering $V_{DD}$ shrinks both margins in absolute volts while the noise on a chip does not shrink with it, which is one of the two limits on how far supply voltages can be scaled.

In [ ]:
def draw_margins(wp_wn, vdd, noise_mV):
    nm = noise_margins(wp_wn, vdd)
    vi, vo = inv_curve(wp_wn, vdd)
    noise = noise_mV * 1e-3
    stages = 12
    v = 0.0
    chain = [v]
    rng = np.random.default_rng(3)
    for i in range(stages):
        v = inv_out(np.clip(v + rng.normal(0, noise), 0, vdd), wp_wn, vdd)
        chain.append(v)
    ratios = np.linspace(0.5, 6, 60)
    nmh = [], []
    NMH = np.array([noise_margins(r, vdd)["NMH"] for r in ratios])
    NML = np.array([noise_margins(r, vdd)["NML"] for r in ratios])

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.25, 1.25, 0.55],
                          wspace=0.3, hspace=0.5, left=0.055, right=0.995,
                          top=0.88, bottom=0.13)

    a0 = panel(fig.add_subplot(gs[:, 0]), BLUE)
    a0.plot(vi, vo, color=POS, lw=2.0)
    a0.plot([0, vdd], [0, vdd], color=GRIDC, lw=0.8, ls="--")
    for x, lab, col in ((nm["VIL"], "$V_{IL}$", DOT), (nm["VIH"], "$V_{IH}$", DOT)):
        a0.axvline(x, color=col, lw=1.0, ls=":")
        a0.text(x, vdd * 0.02, lab, color=col, fontsize=7.5, ha="center")
    for y, lab in ((nm["VOH"], "$V_{OH}$"), (nm["VOL"], "$V_{OL}$")):
        a0.axhline(y, color=ORANGE, lw=1.0, ls=":")
        a0.text(vdd * 0.02, y, lab, color=ORANGE, fontsize=7.5, va="bottom")
    a0.axhspan(nm["VIH"], nm["VOH"], color=GREEN, alpha=0.15)
    a0.axhspan(nm["VOL"], nm["VIL"], color=GREEN, alpha=0.15)
    a0.text(vdd * 0.75, (nm["VIH"] + nm["VOH"]) / 2, f"NMH\n{nm['NMH']:.3f} V",
            color=GREEN, fontsize=8, ha="center", va="center")
    a0.text(vdd * 0.25, (nm["VOL"] + nm["VIL"]) / 2, f"NML\n{nm['NML']:.3f} V",
            color=GREEN, fontsize=8, ha="center", va="center")
    a0.set_xlim(0, vdd); a0.set_ylim(0, vdd)
    a0.set_xlabel("$V_{in}$  (V)"); a0.set_ylabel("$V_{out}$  (V)")
    a0.set_title("the shaded bands are the disturbance a gate absorbs")

    a1 = panel(fig.add_subplot(gs[0, 1]), ORANGE)
    a1.plot(ratios, NMH, color=POS, lw=1.7, label="NMH")
    a1.plot(ratios, NML, color=ORANGE, lw=1.7, label="NML")
    a1.axvline(wp_wn, color=FG, lw=1.0, ls="--")
    a1.axvline(KN0 / KP0, color=DOT, lw=0.9, ls=":")
    a1.text(KN0 / KP0 * 1.05, NMH.max() * 0.4, "symmetric", color=DOT, fontsize=7)
    a1.set_xlabel("$W_p/W_n$"); a1.set_ylabel("volts"); a1.legend(fontsize=7)
    a1.set_title("skew the sizing and one margin shrinks")

    a2 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    a2.step(range(len(chain)), chain, where="mid", color=GREEN, lw=1.6)
    a2.plot(range(len(chain)), chain, "o", ms=5, color=DOT)
    a2.axhline(vdd, color=MUTED, lw=0.8, ls=":")
    a2.axhline(0, color=MUTED, lw=0.8, ls=":")
    a2.axhspan(nm["VIL"], nm["VIH"], color=NEG, alpha=0.12)
    a2.set_ylim(-0.2, vdd + 0.2)
    a2.set_xlabel("gate number"); a2.set_ylabel("volts")
    a2.set_title(f"a chain with ±{noise_mV:.0f} mV injected at every stage")

    readout(fig, 0.845, 0.88, [
        "LEVELS", "─" * 26,
        f"VDD         {vdd:>10.3f}V",
        f"Wp/Wn       {wp_wn:>10.3f}",
        f"VIL         {nm['VIL']:>10.4f}V",
        f"VIH         {nm['VIH']:>10.4f}V",
        f"VOH         {nm['VOH']:>10.4f}V",
        f"VOL         {nm['VOL']:>10.4f}V",
        "", "MARGINS", "─" * 26,
        f"NMH         {nm['NMH']:>10.4f}V",
        f"NML         {nm['NML']:>10.4f}V",
        f"NMH / VDD   {nm['NMH']/vdd*100:>10.1f}%",
        f"NML / VDD   {nm['NML']/vdd*100:>10.1f}%",
        f"asymmetry   {abs(nm['NMH']-nm['NML']):>10.4f}V",
        "", "RESTORATION", "─" * 26,
        f"peak gain   {nm['gain']:>10.2f}",
        f"injected    {noise_mV:>10.1f}mV",
        f"after 12    {chain[-1]:>10.4f}V",
        "CLEAN" if min(chain[-1], vdd - chain[-1]) < 0.02 * vdd else "DEGRADED",
        "", "noise below the margin",
        "is squeezed out, not",
        "passed on",
    ], color=GREEN if noise_mV * 1e-3 < min(nm["NMH"], nm["NML"]) else NEG)
    footer(fig, f"NMH = VOH−VIH = {nm['NMH']:.4f} V   ·   "
                f"NML = VIL−VOL = {nm['NML']:.4f} V   ·   "
                f"boundaries are the unity-gain points")
    plt.show()


w2 = dict(wp_wn=widgets.FloatSlider(value=2.0, min=0.5, max=6.0, step=0.1,
                                    description="Wp/Wn:", **SL),
          vdd=widgets.FloatSlider(value=3.3, min=1.0, max=5.0, step=0.1,
                                  description="VDD (V):", **SL),
          noise_mV=widgets.FloatSlider(value=200, min=0, max=1500, step=25,
                                       description="noise (mV):", **SL))
display(widgets.HBox([w2["wp_wn"], w2["vdd"], w2["noise_mV"]]),
        widgets.interactive_output(draw_margins, w2))

## The crowbar — the current that flows only while switching

Between the two stable states there is a window where **both** devices conduct, and a path exists straight from $V_{DD}$ to ground through two transistors in series. It opens when $V_{in}>V_{TN}$ and closes when $V_{in}>V_{DD}-V_{TP}$:

$$V_{TN}<V_{in}<V_{DD}-V_{TP}$$

At 3.3 V with 0.7 V thresholds that is 0.7 to 2.6 V — **57.6% of the input range** — and the measured peak current is 89.7 µA right at the switching threshold. This is not leakage; it is a genuine short circuit that exists for as long as the input is in transition.

The consequence is a design rule that is not obvious from logic diagrams. Slow input edges keep the crowbar open for longer, so **a gate driven by a slow signal burns more power than the same gate driven by a fast one**, even at identical frequency and load. That is why clock and reset nets get buffered aggressively, and why a floating CMOS input is genuinely dangerous rather than merely undefined — it can park mid-transition and dissipate continuously.

Drag the input edge rate and watch the energy per transition grow. Push $V_{DD}$ down toward $V_{TN}+V_{TP}=1.4$ V and the window closes entirely: below that sum the two devices can never be on together, the crowbar disappears, and this is one of the genuine advantages of low-voltage operation.

In [ ]:
def draw_crowbar(k, vdd, trise_ns, wp_wn, f_MHz):
    vi = np.linspace(0, vdd, 300)
    ic = np.array([crowbar(v, wp_wn, vdd) for v in vi])
    window = max(vdd - VTP - VTN, 0.0)
    n = 500
    t = np.linspace(0, 4 * trise_ns, n)
    ramp = np.clip(t / trise_ns, 0, 1)
    ramp = np.where(t > 2 * trise_ns, np.clip(2 - (t - 2 * trise_ns) / trise_ns, 0, 1),
                    ramp)
    vin_t = ramp * vdd
    ic_t = np.array([crowbar(v, wp_wn, vdd) for v in vin_t])
    E = np.trapezoid(ic_t * vdd, t * 1e-9)
    Psc = E * f_MHz * 1e6
    kk = int(min(k, n - 1))

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.25, 1.25, 0.55],
                          wspace=0.3, hspace=0.5, left=0.055, right=0.995,
                          top=0.88, bottom=0.13)

    a0 = panel(fig.add_subplot(gs[:, 0]), BLUE)
    a0.plot(vi, ic * 1e6, color=NEG, lw=2.0)
    a0.fill_between(vi, 0, ic * 1e6, color=NEG, alpha=0.25)
    if window > 0:
        a0.axvspan(VTN, vdd - VTP, color=NEG, alpha=0.08)
        a0.text((VTN + vdd - VTP) / 2, ic.max() * 1e6 * 0.85,
                "both devices on", color=NEG, fontsize=8, ha="center")
    a0.axvline(vin_t[kk], color=FG, lw=1.0, ls=":")
    a0.plot([vin_t[kk]], [ic_t[kk] * 1e6], "o", ms=7, color=DOT)
    a0.set_xlabel("$V_{in}$  (V)"); a0.set_ylabel("supply current  (µA)")
    if window > 0:
        a0.set_title(f"peak {ic.max()*1e6:.2f} µA  ·  window {window:.2f} V "
                     f"= {window/vdd*100:.1f}% of the range")
    else:
        a0.set_title(f"no window at all — VDD = {vdd:.2f} V < "
                     f"VTN+VTP = {VTN+VTP:.2f} V")

    a1 = panel(fig.add_subplot(gs[0, 1]), ORANGE)
    a1.plot(t, vin_t, color=POS, lw=1.5, label="$V_{in}$")
    a1b = a1.twinx()
    a1b.fill_between(t, 0, ic_t * 1e6, color=NEG, alpha=0.35)
    a1b.plot(t, ic_t * 1e6, color=NEG, lw=1.2)
    a1b.set_ylabel("µA", color=NEG); a1b.tick_params(colors=NEG, labelsize=7)
    a1b.grid(False)
    a1.axvline(t[kk], color=FG, lw=1.0, ls=":")
    a1.set_ylabel("volts", color=POS); a1.legend(fontsize=7, loc="upper right")
    a1.set_title(f"one transition costs {E*1e15:.2f} fJ")

    a2 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    trs = np.logspace(-1.3, 1.3, 40)
    Es = []
    for tr in trs:
        tt = np.linspace(0, 2 * tr, 200)
        vv = np.clip(tt / tr, 0, 1) * vdd
        ii = np.array([crowbar(v, wp_wn, vdd) for v in vv])
        Es.append(np.trapezoid(ii * vdd, tt * 1e-9))
    Es = np.array(Es)
    if Es.max() > 0:
        a2.loglog(trs, np.maximum(Es * 1e15, 1e-9), color=GREEN, lw=1.8)
        a2.set_ylabel("energy per edge  (fJ)")
        a2.set_title("slow edges cost more — proportional to the transition time")
    else:
        a2.set_xscale("log")
        a2.plot(trs, np.zeros_like(trs), color=GREEN, lw=2.0)
        a2.set_ylim(-1, 1); a2.set_yticks([0])
        a2.set_ylabel("energy per edge  (fJ)")
        a2.set_title(f"exactly zero — VDD = {vdd:.2f} V is below "
                     f"VTN+VTP = {VTN+VTP:.2f} V")
    a2.axvline(trise_ns, color=FG, lw=1.0, ls="--")
    a2.set_xlabel("input rise time  (ns)")

    readout(fig, 0.845, 0.88, [
        "PROCESS", "─" * 26,
        f"VDD         {vdd:>10.2f}V",
        f"VTN + VTP   {VTN+VTP:>10.2f}V",
        f"Wp/Wn       {wp_wn:>10.2f}",
        "", "WINDOW", "─" * 26,
        f"opens at    {VTN:>10.2f}V",
        f"closes at   {max(vdd-VTP,VTN):>10.2f}V",
        f"width       {window:>10.3f}V",
        f"of range    {window/vdd*100:>10.1f}%",
        "", "CURRENT", "─" * 26,
        f"peak        {ic.max()*1e6:>10.3f}µA",
        f"at Vin      {vi[int(np.argmax(ic))]:>10.3f}V",
        "", "ENERGY", "─" * 26,
        f"rise time   {trise_ns:>10.3f}ns",
        f"per edge    {E*1e15:>10.4f}fJ",
        f"at {f_MHz:.0f} MHz   {Psc*1e6:>10.4f}µW",
        "", ("below VDD = VTN+VTP" if window > 0 else "NO CROWBAR AT ALL"),
        ("the window closes and" if window > 0 else "VDD is below VTN+VTP"),
        ("the crowbar vanishes" if window > 0 else "so both can never"),
        ("" if window > 0 else "conduct together"),
    ], color=NEG if window > 0 else GREEN)
    footer(fig, f"short-circuit path exists for VTN < Vin < VDD−VTP   ·   "
                f"energy ∝ transition time   ·   buffer your slow nets")
    plt.show()


_p3, _s3 = timeline(99, step=2)
w3 = dict(vdd=widgets.FloatSlider(value=3.3, min=1.0, max=5.0, step=0.1,
                                  description="VDD (V):", **SL),
          trise_ns=widgets.FloatSlider(value=1.0, min=0.1, max=10, step=0.1,
                                       description="rise time ns:", **SL),
          wp_wn=widgets.FloatSlider(value=2.0, min=0.5, max=6.0, step=0.1,
                                    description="Wp/Wn:", **SL),
          f_MHz=widgets.FloatSlider(value=100, min=1, max=2000, step=10,
                                    description="f (MHz):", **SL),
          k=_s3)
display(widgets.VBox([widgets.HBox([w3["vdd"], w3["trise_ns"], w3["wp_wn"]]),
                      widgets.HBox([w3["f_MHz"], _p3, _s3])]),
        widgets.interactive_output(draw_crowbar, w3))

## Delay — a gate is an RC circuit charging the next gate

Nothing in a logic diagram suggests time, but every gate output has to charge the input capacitance of everything it drives, through the on-resistance of a transistor. That is the RC from the very first circuits notebook, and it is the whole story of digital speed:

$$R_{on}\approx\frac{1}{k(V_{DD}-V_{th})},\qquad
t_p\approx0.69\,R_{on}C_L$$

The load is not a component you placed — it is the **gates you are driving**. Each one adds its own gate capacitance, so delay grows linearly with fan-out: measured 3.98 ps at a fan-out of 1, rising to 22.6 ps at a fan-out of 8 for the same driver.

Two knobs make it faster and each has a catch. Making the driver wider lowers $R_{on}$ proportionally — but a wider driver has a larger input capacitance, so it loads *its* driver more, which is why buffer chains grow in stages rather than jumping straight to a big transistor. Raising $V_{DD}$ increases the drive current — and costs power quadratically, which the next section deals with.

The waveform panel shows what a "logic level" really looks like mid-flight: an exponential, not a square edge, and the gate downstream does not decide anything until that exponential crosses its threshold. Stack enough of these and you have the critical path that sets a clock frequency.

In [ ]:
def gate_delay(vdd, W, CL, kn=KN0):
    Ron = 1.0 / (kn * W * (vdd - VTN))
    return 0.69 * Ron * CL, Ron


def draw_delay(k, vdd, W, fanout, Cg_fF, Cwire_fF):
    Cg, Cw = Cg_fF * 1e-15, Cwire_fF * 1e-15
    CL = fanout * Cg + Cw
    tp, Ron = gate_delay(vdd, W, CL)
    n = 600
    t = np.linspace(0, 6 * tp, n)
    v = vdd * (1 - np.exp(-t / (Ron * CL)))
    kk = int(min(k, n - 1))
    fos = np.arange(1, 13)
    tps = np.array([gate_delay(vdd, W, f * Cg + Cw)[0] for f in fos])

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.15, 1.25, 0.55],
                          wspace=0.3, hspace=0.46, left=0.02, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.6, 4.6), (-0.5, 3.4))
    dp, sp = mos_switch(a0, (1.5, 2.4), ptype=True)
    dn, sn = mos_switch(a0, (1.5, 1.0), ptype=False)
    wire(a0, [(1.5 + 0.25, 2.4 + 0.54), (1.75, 3.1)], vdd, vdd)
    wire(a0, [(1.75, 3.1), (0.5, 3.1)], vdd, vdd)
    wire(a0, [(1.5 + 0.25, 2.4 - 0.54), (1.75, 1.0 + 0.54)], v[kk], vdd)
    wire(a0, [(1.5 + 0.25, 1.0 - 0.54), (1.75, 0.2)], 0.0, vdd)
    wire(a0, [(0.5, 0.2), (4.2, 0.2)], 0.0, vdd)
    wire(a0, [(1.75, 1.7), (3.0, 1.7)], v[kk], vdd)
    node_dot(a0, (3.0, 1.7), v[kk], vdd)
    for i in range(int(fanout)):
        y = 1.7 + (i - (fanout - 1) / 2) * 0.42
        wire(a0, [(3.0, 1.7), (3.4, y), (3.9, y)], v[kk], vdd)
        a0.add_patch(mpatches.Polygon([[3.9, y - 0.16], [3.9, y + 0.16],
                                       [4.25, y]], closed=True, fill=False,
                                      ec=MUTED, lw=1.4))
    capacitor(a0, (3.0, 1.7), (3.0, 0.2), v[kk] / 2, vdd, f"CL")
    source(a0, (0.5, 0.2), (0.5, 3.1), vdd / 2, vdd, "dc", f"{vdd:.1f}V")
    a0.text(4.35, 1.7, f"×{int(fanout)}", color=FG, fontsize=8, va="center")
    charge_dots(a0, seg((1.75, 3.1), (3.0, 1.7), 24),
                k / 100 * (vdd - v[kk]) / Ron * 2e6, spacing=0.22, ms=3.4)
    a0.set_title(f"charging {CL*1e15:.1f} fF through {Ron:.0f} Ω", fontsize=8.5)

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a1.plot(t * 1e12, v, color=POS, lw=1.8)
    a1.axhline(vdd / 2, color=DOT, lw=1.0, ls="--")
    a1.axvline(tp * 1e12, color=DOT, lw=1.0, ls=":")
    a1.text(tp * 1e12 * 1.1, vdd * 0.25, f"$t_p$ = {tp*1e12:.2f} ps",
            color=DOT, fontsize=8)
    a1.axvline(t[kk] * 1e12, color=FG, lw=1.0, ls=":")
    a1.plot([t[kk] * 1e12], [v[kk]], "o", ms=7, color=ORANGE)
    a1.set_ylabel("output  (V)")
    a1.set_title("a logic edge is an exponential, not a step")

    a2 = panel(fig.add_subplot(gs[1, 1]), ORANGE)
    a2.plot(fos, tps * 1e12, "o-", color=ORANGE, lw=1.7, ms=5)
    a2.axvline(fanout, color=FG, lw=1.0, ls="--")
    a2.set_xlabel("fan-out  (gates driven)")
    a2.set_ylabel("delay  (ps)")
    a2.set_title("delay grows linearly with the number of loads")

    readout(fig, 0.845, 0.90, [
        "DRIVER", "─" * 26,
        f"VDD         {vdd:>10.2f}V",
        f"width W     {W:>10.2f}×",
        f"Ron         {Ron:>10.1f}Ω",
        "", "LOAD", "─" * 26,
        f"fan-out     {int(fanout):>10d}",
        f"gate C each {Cg_fF:>10.2f}fF",
        f"wire C      {Cwire_fF:>10.2f}fF",
        f"total CL    {CL*1e15:>10.2f}fF",
        "", "TIMING", "─" * 26,
        f"RC          {Ron*CL*1e12:>10.3f}ps",
        f"tp = 0.69RC {tp*1e12:>10.3f}ps",
        f"10-90% rise {2.2*Ron*CL*1e12:>10.3f}ps",
        f"max freq    {1/(2*tp)/1e9:>10.2f}GHz",
        "", "REFERENCE", "─" * 26,
        "fanout 1 →  3.98 ps",
        "fanout 4 → 11.94 ps",
        "fanout 8 → 22.56 ps",
        "", "wider driver is faster",
        "but loads its own driver",
    ])
    footer(fig, f"tp = 0.69·Ron·CL   ·   Ron = 1/(kW(VDD−Vth))   ·   "
                f"the load is the gates you drive")
    plt.show()


_p4, _s4 = timeline(99, step=2)
w4 = dict(vdd=widgets.FloatSlider(value=3.3, min=0.8, max=5.0, step=0.1,
                                  description="VDD (V):", **SL),
          W=widgets.FloatSlider(value=1.0, min=0.25, max=8.0, step=0.25,
                                description="driver W:", **SL),
          fanout=widgets.IntSlider(value=4, min=1, max=12, step=1,
                                   description="fan-out:", **SL),
          Cg_fF=widgets.FloatSlider(value=2.0, min=0.5, max=10, step=0.5,
                                    description="gate C (fF):", **SL),
          Cwire_fF=widgets.FloatSlider(value=1.0, min=0, max=50, step=1,
                                       description="wire C (fF):", **SL),
          k=_s4)
display(widgets.VBox([widgets.HBox([w4["vdd"], w4["W"], w4["fanout"]]),
                      widgets.HBox([w4["Cg_fF"], w4["Cwire_fF"]]),
                      widgets.HBox([_p4, _s4])]),
        widgets.interactive_output(draw_delay, w4))

## Power — the equation that ended the megahertz race

Every transition moves charge $CV_{DD}$ onto or off a node, and half the energy drawn from the supply is dissipated in the switch regardless of how good the switch is. Averaged over many cycles:

$$P_{dyn}=\alpha C V_{DD}^2 f$$

The **square** on the voltage is the important part. Dropping from 5 V to 1.8 V cuts power to 13% of what it was, at identical capacitance and frequency — no other term in the equation offers anything like that leverage, which is why supply voltages fell from 5 V through 3.3, 1.8, 1.2 and below.

But the same voltage sets the drive current, and the current sets the speed. With $I\propto(V_{DD}-V_{th})^2$ and $t_p\propto CV_{DD}/I$, lowering the supply makes the gate slower — measured 2.44 ns at 3.3 V against 55.6 ns at 1.0 V, a 23× slowdown for a 10.9× power saving. There is an optimum, and it is not at either extreme.

The energy–delay product panel is where the trade becomes a single number. Minimising $P\cdot t_p$ rather than either alone is what modern designs actually target, and it explains the shape of the last two decades: clock frequencies stopped rising around 2005 not because transistors stopped improving but because $P=\alpha CV^2f$ made further frequency increases unaffordable, and the industry went sideways into multiple cores instead.

Static power is negligible here — dynamic exceeds leakage by a factor of $3\times10^6$ at 1 GHz in this model. In deep-submicron processes that ratio collapses, and leakage becomes the reason parts of a chip are switched off entirely.

In [ ]:
def draw_power(vdd, C_fF, f_MHz, alpha, Ileak_nA):
    C, f, Ileak = C_fF * 1e-15, f_MHz * 1e6, Ileak_nA * 1e-9
    Pdyn = alpha * C * vdd ** 2 * f
    Pstat = Ileak * vdd
    I = 0.5 * KN0 * max(vdd - VTN, 1e-6) ** 2
    tp = C * vdd / (2 * I)
    EDP = Pdyn * tp
    vv = np.linspace(0.9, 5.0, 300)
    Pv = alpha * C * vv ** 2 * f
    Iv = 0.5 * KN0 * np.maximum(vv - VTN, 1e-6) ** 2
    tpv = C * vv / (2 * Iv)
    edp = Pv * tpv
    vopt = vv[int(np.argmin(edp))]

    fig = plt.figure(figsize=(13.0, 4.9))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.25, 1.25, 0.55],
                          wspace=0.3, hspace=0.5, left=0.055, right=0.995,
                          top=0.88, bottom=0.13)

    a0 = panel(fig.add_subplot(gs[:, 0]), BLUE)
    a0.plot(vv, Pv * 1e3, color=POS, lw=2.0, label="dynamic  αCV²f")
    a0.axhline(Pstat * 1e3, color=ORANGE, lw=1.4, ls="--", label="static  I·V")
    a0.axvline(vdd, color=FG, lw=1.0, ls=":")
    a0.plot([vdd], [Pdyn * 1e3], "o", ms=8, color=DOT)
    for vx in (1.0, 1.8, 3.3, 5.0):
        a0.plot([vx], [alpha * C * vx ** 2 * f * 1e3], "o", ms=4, color=MUTED)
        a0.text(vx, alpha * C * vx ** 2 * f * 1e3 * 1.12, f"{vx}V", color=MUTED,
                fontsize=6.5, ha="center")
    a0.set_yscale("log")
    a0.set_xlabel("$V_{DD}$  (V)"); a0.set_ylabel("power  (mW)")
    a0.legend(fontsize=7.5)
    a0.set_title(f"quadratic in voltage — 5 V → 1.8 V is "
                 f"{1.8**2/5**2*100:.0f}% of the power")

    a1 = panel(fig.add_subplot(gs[0, 1]), ORANGE)
    a1.plot(vv, tpv * 1e12, color=ORANGE, lw=1.8)
    a1.axvline(vdd, color=FG, lw=1.0, ls=":")
    a1.plot([vdd], [tp * 1e12], "o", ms=7, color=DOT)
    a1.set_yscale("log"); a1.set_ylabel("delay  (ps)")
    a1.set_title("but the same voltage sets the speed")

    a2 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    a2.plot(vv, edp / edp.min(), color=GREEN, lw=1.8)
    a2.axvline(vopt, color=DOT, lw=1.2, ls="--")
    a2.text(vopt * 1.03, 2.5, f"min at {vopt:.2f} V", color=DOT, fontsize=7.5)
    a2.axvline(vdd, color=FG, lw=1.0, ls=":")
    a2.set_ylim(0.8, 8)
    a2.set_xlabel("$V_{DD}$  (V)"); a2.set_ylabel("P·t  (normalised)")
    a2.set_title("energy–delay product — the quantity actually minimised")

    readout(fig, 0.845, 0.88, [
        "OPERATING", "─" * 26,
        f"VDD         {vdd:>10.2f}V",
        f"C           {C_fF:>10.2f}fF",
        f"frequency   {f_MHz:>10.1f}MHz",
        f"activity α  {alpha:>10.3f}",
        "", "POWER", "─" * 26,
        f"dynamic     {Pdyn*1e6:>10.4f}µW",
        f"static      {Pstat*1e6:>10.6f}µW",
        f"ratio       {Pdyn/max(Pstat,1e-18):>10.0f}",
        f"per edge    {alpha*C*vdd**2*1e15:>10.4f}fJ",
        "", "SPEED", "─" * 26,
        f"drive I     {I*1e6:>10.3f}µA",
        f"delay       {tp*1e12:>10.2f}ps",
        f"max f       {1/(2*tp)/1e9:>10.3f}GHz",
        "", "TRADE", "─" * 26,
        f"P·t         {EDP*1e15:>10.4f}fJ",
        f"optimum VDD {vopt:>10.3f}V",
        "", "at 5 V → 1.8 V:",
        "power ×0.13, delay ×2.4",
        "which is why clocks",
        "stopped getting faster",
    ], color=NEG if Pdyn > 1e-3 else GREEN)
    footer(fig, f"P = αCV²f = {Pdyn*1e6:.3f} µW   ·   "
                f"delay ∝ CV/(V−Vth)²   ·   minimise the product, not either one")
    plt.show()


w5 = dict(vdd=widgets.FloatSlider(value=3.3, min=0.9, max=5.0, step=0.1,
                                  description="VDD (V):", **SL),
          C_fF=widgets.FloatSlider(value=20, min=1, max=200, step=1,
                                   description="C (fF):", **SL),
          f_MHz=widgets.FloatSlider(value=500, min=1, max=5000, step=10,
                                    description="f (MHz):", **SL),
          alpha=widgets.FloatSlider(value=0.15, min=0.01, max=1.0, step=0.01,
                                    description="activity α:", **SL),
          Ileak_nA=widgets.FloatSlider(value=1.0, min=0.001, max=1000, step=1,
                                       description="leakage (nA):", **SL))
display(widgets.HBox([w5["vdd"], w5["C_fF"], w5["f_MHz"], w5["alpha"],
                      w5["Ileak_nA"]]),
        widgets.interactive_output(draw_power, w5))

## Memory — two inverters that hold each other's opinion

Nothing so far can remember anything. Connect two inverters in a ring and the circuit acquires **state**: whatever the first one outputs, the second inverts back into it, so the pair reinforces its own condition indefinitely.

The panel finds the fixed points of $v=\text{inv}(\text{inv}(v))$ numerically. There are exactly three, and only two are stable: the two rails. The one in the middle at $V_{DD}/2$ is a genuine equilibrium — the equations balance there — but the loop gain around the ring exceeds one, so any deviation is amplified rather than corrected. Started at $V_{DD}/2+1\ \mu\text{V}$ the simulation runs to the rail within a handful of iterations.

That instability is exactly what makes the cell useful. A memory that gently returned to a middle value would be useless; one that violently commits to whichever side it was pushed toward is a bit.

It also introduces the one failure mode with no design fix. If the latch is closed at the instant its input is changing — the **setup and hold** window — it can be left arbitrarily close to that middle point, and the time it takes to resolve is unbounded. That is **metastability**, and the resolution time is exponential:

$$P(\text{unresolved after }t)\propto e^{-t/\tau}$$

You cannot eliminate it, only make it improbable: wait longer before believing the output. The panel simulates the pair **in time**, with the node capacitance included, and measures the resolution time as the starting offset is reduced. The result is a straight line on a log axis at **60.8 ps per decade** — every factor of ten closer to $V_M$ costs the same fixed extra wait, which is exactly the exponential law above with $\tau$ measurable rather than assumed.

That is where digital design stops being about logic and becomes about probability: a synchroniser is just a second flip-flop giving the first one another clock period to resolve, and the specification is a mean time between failures, not a guarantee.

In [ ]:
def inv_inet(vin, vout, wp_wn=2.0, vdd=VDD):
    """Net current charging the output node of one inverter."""
    return (sq_id(vdd - vin, vdd - vout, KP0 * wp_wn, VTP)
            - sq_id(vin, vout, KN0, VTN))


def latch_sim(offset, wp_wn=2.0, vdd=VDD, C=10e-15, dt=2e-13, nmax=6000):
    """Time-domain cross-coupled pair — the node capacitance is what makes
    metastability take *time* rather than resolving instantly."""
    Vm = switch_threshold(wp_wn, vdd)
    va, vb = Vm + offset, Vm
    ta, tb = [va], [vb]
    res = nmax
    for k in range(nmax):
        ia = inv_inet(vb, va, wp_wn, vdd)
        ib = inv_inet(va, vb, wp_wn, vdd)
        va = float(np.clip(va + ia / C * dt, 0, vdd))
        vb = float(np.clip(vb + ib / C * dt, 0, vdd))
        ta.append(va); tb.append(vb)
        if res == nmax and abs(va - Vm) > 0.45 * vdd:
            res = k
    return np.array(ta), np.array(tb), Vm, res, dt


def draw_latch(k, offset_uV, wp_wn, vdd, C_fF):
    C = C_fF * 1e-15
    ta, tb, Vm, res, dt = latch_sim(offset_uV * 1e-6, wp_wn, vdd, C)
    t = np.arange(len(ta)) * dt
    kk = int(min(k / 60 * len(ta), len(ta) - 1))
    vi, vo = inv_curve(wp_wn, vdd)
    offs = np.logspace(-2, -7, 6)
    times = []
    for o in offs:
        _, _, _, r, _ = latch_sim(o, wp_wn, vdd, C)
        times.append(r * dt)
    times = np.array(times)

    fig = plt.figure(figsize=(13.0, 5.2))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.1, 1.3, 0.55],
                          wspace=0.3, hspace=0.46, left=0.02, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.6, 4.6), (-0.6, 2.8))
    va, vb = ta[kk], tb[kk]
    for x, vv_ in ((1.4, va), (3.0, vb)):
        a0.add_patch(mpatches.Polygon([[x - 0.45, 0.9], [x - 0.45, 2.1],
                                       [x + 0.45, 1.5]], closed=True,
                                      fill=False, ec=FG, lw=2.0))
        a0.add_patch(mpatches.Circle((x + 0.58, 1.5), 0.11, fill=False, ec=FG,
                                     lw=1.6))
    wire(a0, [(1.98, 1.5), (2.55, 1.5)], vb, vdd)
    wire(a0, [(3.58, 1.5), (4.1, 1.5), (4.1, 2.5), (1.0, 2.5), (1.0, 1.5),
              (0.95, 1.5)], va, vdd)
    capacitor(a0, (2.3, 1.5), (2.3, 0.3), vb / 2, vdd, f"{C_fF:.0f}fF")
    capacitor(a0, (4.1, 1.5), (4.1, 0.3), va / 2, vdd, f"{C_fF:.0f}fF")
    wire(a0, [(2.3, 0.3), (4.1, 0.3)], 0.0, vdd)
    node_dot(a0, (2.3, 1.5), vb, vdd)
    node_dot(a0, (4.1, 1.5), va, vdd)
    a0.text(2.3, 1.75, f"B {vb:.4f}", color=FG, fontsize=7.5, ha="center")
    a0.text(4.1, 1.75, f"A {va:.4f}", color=FG, fontsize=7.5, ha="center")
    a0.set_title(f"t = {t[kk]*1e12:.1f} ps", fontsize=8.5)

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a1.plot(t * 1e12, ta, color=POS, lw=1.7, label="A")
    a1.plot(t * 1e12, tb, color=PURP, lw=1.7, label="B")
    a1.axhline(Vm, color=NEG, lw=1.0, ls="--")
    a1.text(t[-1] * 1e12 * 0.02, Vm * 1.03, "$V_M$", color=NEG, fontsize=7.5)
    a1.axvline(t[kk] * 1e12, color=FG, lw=1.0, ls=":")
    a1.axvline(res * dt * 1e12, color=DOT, lw=1.0, ls="--")
    a1.set_ylabel("volts"); a1.legend(fontsize=7)
    a1.set_title(f"resolved after {res*dt*1e12:.1f} ps")

    a2 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    a2.semilogx(offs * 1e6, times * 1e12, "o-", color=GREEN, lw=1.8, ms=5)
    a2.axvline(offset_uV, color=FG, lw=1.0, ls="--")
    slope = np.polyfit(np.log10(offs), times * 1e12, 1)[0]
    a2.set_xlabel("initial offset  (µV)")
    a2.set_ylabel("resolution time  (ps)")
    a2.set_title(f"a straight line on a log axis — {abs(slope):.1f} ps per decade")

    readout(fig, 0.845, 0.90, [
        "CELL", "─" * 26,
        f"VDD         {vdd:>10.2f}V",
        f"Wp/Wn       {wp_wn:>10.2f}",
        f"node C      {C_fF:>10.1f}fF",
        f"VM          {Vm:>10.5f}V",
        "", "EQUILIBRIA", "─" * 26,
        f"low         {0.0:>10.4f}V stable",
        f"metastable  {Vm:>10.4f}V UNSTABLE",
        f"high        {vdd:>10.4f}V stable",
        "", "THIS RUN", "─" * 26,
        f"offset      {offset_uV:>10.4f}µV",
        f"time now    {t[kk]*1e12:>10.2f}ps",
        f"vA          {va:>10.5f}V",
        f"deviation   {abs(va-Vm):>10.3e}V",
        f"resolved at {res*dt*1e12:>10.2f}ps",
        "", "SCALING", "─" * 26,
        f"per decade  {abs(slope):>10.2f}ps",
        f"10× smaller +{abs(slope):>9.2f}ps",
        "", "you cannot remove it,",
        "only wait longer and",
        "make it improbable",
    ], color=NEG if t[kk] * 1e12 < res * dt * 1e12 else GREEN)
    footer(fig, "two inverters, three equilibria, two stable   ·   "
                f"resolution time grows logarithmically — {abs(slope):.1f} ps "
                f"per decade of offset")
    plt.show()


_p6, _s6 = timeline(59, step=1, interval=90)
w6 = dict(offset_uV=widgets.FloatSlider(value=1.0, min=0.01, max=10000, step=0.01,
                                        description="offset (µV):",
                                        readout_format=".2f", **SL),
          wp_wn=widgets.FloatSlider(value=2.0, min=0.5, max=6.0, step=0.1,
                                    description="Wp/Wn:", **SL),
          vdd=widgets.FloatSlider(value=3.3, min=1.5, max=5.0, step=0.1,
                                  description="VDD (V):", **SL),
          C_fF=widgets.FloatSlider(value=10, min=1, max=50, step=1,
                                   description="node C (fF):", **SL),
          k=_s6)
display(widgets.VBox([widgets.HBox([w6["offset_uV"], w6["wp_wn"], w6["vdd"]]),
                      widgets.HBox([w6["C_fF"], _p6, _s6])]),
        widgets.interactive_output(draw_latch, w6))